## 03-sql.ipynb
1. 
2. 질문과 관련된 테이블을 LLM이 결정
3. 해당 테이블의 스키마 확인하기.
4. 
5. LLM을 사용하여 흔히 발생하는 오류가 있는지 SQL 확인
6. DB에서 SQL을 실행하고 결과를 확인
7. DB에서 에러 발생시, 수정 후 다시 확인
8. DB 결과를 바탕으로 LLM이 답변 생성


In [ ]:
# %pip install -U langgraph
# %pip install -U sqlalchemy psycopg2-binary

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_community.utilities import SQLDatabase
import os

DB_URI = os.environ.get('DB_URI')

db = SQLDatabase.from_uri(DB_URI)

In [ ]:
#print(db.get_table_info())
#print(db.get_usable_table_names())
print(db.run('SELECT * FROM sales LIMIT 5;'))

In [ ]:
# LLM 초기화
from langchain_openai import ChatOpenAI
model = ChatOpenAI(name='gpt-4.1-mini')

In [ ]:
# Agent 용 Tool 만들기
from langchain_community.agent_toolkits import SQLDatabaseToolkit
toolkit = SQLDatabaseToolkit(db=db, llm=model)

In [ ]:
# Agent 만들기
from langchain.agents import create_agent

dialect = db.dialect
top_k = 5 # 5개만 가져와라
system_prompt = f"""
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
"""

agent = create_agent(model, toolkit.get_tools(), system_prompt=system_prompt)

for tool in toolkit.get_tools():
    print(tool.name, ':', tool.description)
#sql_db_query, sql_db_schema, sql_db_list_tables, sql_db_query_checker

In [28]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
#%pip install -U langgraph
from langgraph.checkpoint.memory import InMemorySaver #중단했다가 다시 시작하기 위해 필요한 포인트 저장

agent = create_agent(
    model,
    toolkit.get_tools(),
    system_prompt=system_prompt,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={'sql_db_query': True},
            description_prefix='Tool 실행 전에 승인을 기다림'
        )
    ],
    checkpointer=InMemorySaver() # 일시정지 - 재실행에서 돌아갈 곳을 기억해야함.
)

In [ ]:
from langgraph.types import Command
#question = '전체 평균 매출액과, 가장 구매를 많이한 손님 3명의 이름을 알려줘'
question = '2월에 가장 많이 팔린 물건 3개와, 해당 물건들의 토요일 일요일 평균 매출액'

config = {'configurable': {'thread_id': '1234567'}} # 돌릴 때마다 thread_id를 바꿔줘야 함.

for event in agent.stream(
    {'messages': [{'role': 'user', 'content': question}]},
    stream_mode='values',
    config=config,
):
    if "__interrupt__" in event: 
        print("INTERRUPTED:") 
        interrupt = event["__interrupt__"][0] 
        for request in interrupt.value["action_requests"]: 
            print(request["description"]) 
    elif "messages" in event:
        event["messages"][-1].pretty_print()
    else:
        pass

print('-------------------------중단 이후----------------------------------------')

for step in agent.stream(
    Command(resume={"decisions": [{"type": "approve"}]}), 
    config,
    stream_mode="values",
):
    if "messages" in step:
        step["messages"][-1].pretty_print()
    elif "__interrupt__" in step:
        print("INTERRUPTED:")
        interrupt = step["__interrupt__"][0]
        for request in interrupt.value["action_requests"]:
            print(request["description"])
    else:
        pass


================================ Human Message =================================

2월에 가장 많이 팔린 물건 3개와, 해당 물건들의 토요일 일요일 평균 매출액
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_WfnFKteNBlKVqPiR6qWDW3g9)
 Call ID: call_WfnFKteNBlKVqPiR6qWDW3g9
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

courses, customers, sales, students, students_courses
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_wUobMHRQT9DK4niyVr9FXCi2)
 Call ID: call_wUobMHRQT9DK4niyVr9FXCi2
  Args:
    table_names: sales
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE sales (
	id INTEGER NOT NULL, 
	order_date DATE NOT NULL, 
	customer_id VARCHAR(10) NOT NULL, 
	product_id VARCHAR(10) NOT NULL, 
	product_name VARCHAR(50) NOT NULL, 
	category VARCHAR(20) NO